### Conhecendo o dataset SLCE3

In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error


# Carrega o dataset
df = pd.read_csv('SLCE3.csv')

print("Primeiras 5 linhas do DataFrame:")
print(df.head())

print("\nInformações gerais do DataFrame:")
print(df.info())

Primeiras 5 linhas do DataFrame:
         Date     Close      High       Low      Open   Volume
0  2018-01-02  3.077934  3.103537  2.972182  2.972182  1227908
1  2018-01-03  3.104650  3.215968  3.067915  3.077933  2494536
2  2018-01-04  3.092405  3.198157  3.055670  3.104650  2957724
3  2018-01-05  3.150290  3.166988  3.069028  3.138046  2140248
4  2018-01-08  3.133592  3.183685  3.112442  3.150290  1613172

Informações gerais do DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1738 entries, 0 to 1737
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    1738 non-null   object 
 1   Close   1738 non-null   float64
 2   High    1738 non-null   float64
 3   Low     1738 non-null   float64
 4   Open    1738 non-null   float64
 5   Volume  1738 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 81.6+ KB
None


### Tratamento do dataset para valores nulos

In [25]:
# Remove linhas com qualquer valor nulo
df = df.dropna()
print("Após remoção de linhas com valores nulos:")
print(df.info())

Após remoção de linhas com valores nulos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1738 entries, 0 to 1737
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    1738 non-null   object 
 1   Close   1738 non-null   float64
 2   High    1738 non-null   float64
 3   Low     1738 non-null   float64
 4   Open    1738 non-null   float64
 5   Volume  1738 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 81.6+ KB
None


### Criação das janelas deslizantes de Close futuro

In [26]:
# Criar janelas deslizantes de Close futuro (3, 7, 15 e 30 dias)
df['Close_3d_fut'] = df['Close'].shift(-3)
df['Close_7d_fut'] = df['Close'].shift(-7)  
df['Close_15d_fut'] = df['Close'].shift(-15)
df['Close_30d_fut'] = df['Close'].shift(-30)

print("Janelas deslizantes de Close futuro criadas:")
print("- Close_3d_fut: Close 3 dias no futuro")
print("- Close_7d_fut: Close 7 dias no futuro") 
print("- Close_15d_fut: Close 15 dias no futuro")
print("- Close_30d_fut: Close 30 dias no futuro")

# Remover linhas com valores nulos nas colunas futuras
df_clean = df.dropna(subset=['Close_3d_fut', 'Close_7d_fut', 'Close_15d_fut', 'Close_30d_fut']).copy()

print(f"\nDataset após remoção de nulos: {df_clean.shape[0]} linhas")

Janelas deslizantes de Close futuro criadas:
- Close_3d_fut: Close 3 dias no futuro
- Close_7d_fut: Close 7 dias no futuro
- Close_15d_fut: Close 15 dias no futuro
- Close_30d_fut: Close 30 dias no futuro

Dataset após remoção de nulos: 1708 linhas


### Criação de variáveis dummy e aplicação do OneHotEncoder

In [ ]:
# Converte a coluna Date para datetime
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

# Cria a variável dummy categórica baseada nos períodos identificados
# 0: antes de 2021-05-05 (período inicial)
# 1: entre 2021-05-05 e 2022-02-07 (período de alta volatilidade)
# 2: após 2022-02-07 (período de estabilização)
df_clean['dummy_period'] = 0
df_clean.loc[df_clean['Date'] >= pd.to_datetime('2021-05-05'), 'dummy_period'] = 1
df_clean.loc[df_clean['Date'] >= pd.to_datetime('2022-02-07'), 'dummy_period'] = 2

print(f"Distribuição da variável dummy_period:")
print(df_clean['dummy_period'].value_counts().sort_index())

# Aplicar OneHotEncoder do scikit-learn
encoder = OneHotEncoder(drop='first', sparse_output=False)
dummy_encoded = encoder.fit_transform(df_clean[['dummy_period']])

# Criar DataFrame com as colunas dummy encodadas
dummy_columns = [f'period_{int(cat)}' for cat in encoder.categories_[0][1:]]  # Exclui primeira categoria
dummy_df = pd.DataFrame(dummy_encoded, columns=dummy_columns, index=df_clean.index)

# Concatenar com DataFrame original
df_final = pd.concat([df_clean, dummy_df], axis=1)
print(f"\nColunas criadas pelo OneHotEncoder: {dummy_columns}")
print(f"DataFrame final tem {df_final.shape[0]} linhas e {df_final.shape[1]} colunas")

Distribuição da variável dummy_period:
dummy_period
0    824
1    190
2    694
Name: count, dtype: int64

Colunas criadas pelo OneHotEncoder: ['period_1', 'period_2']
DataFrame final tem 1708 linhas e 13 colunas


### Salvando dataset com alterações feitas

In [ ]:
df_final.to_csv('SLCE3_tratado.csv', index=False)

### Modelos sem variáveis dummy (apenas Low, High, Open)

In [28]:
# Definir variáveis independentes (sem dummy)
X_numeric = df_final[['Low', 'High', 'Open']]

# Definir targets (variáveis dependentes)
targets = {
    '3d': df_final['Close_3d_fut'],
    '7d': df_final['Close_7d_fut'], 
    '15d': df_final['Close_15d_fut'],
    '30d': df_final['Close_30d_fut']
}

print("MODELOS SEM VARIÁVEIS DUMMY")
print("=" * 50)

# Dicionário para armazenar modelos treinados
models_numeric = {}

# Treinar modelo para cada janela temporal
for period, y in targets.items():
    # Divisão treino-teste
    X_train, X_test, y_train, y_test = train_test_split(
        X_numeric, y, test_size=0.2, random_state=42)
    
    # Treinar modelo
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Fazer predições
    y_pred = model.predict(X_test)
    
    # Calcular métricas
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Armazenar modelo
    models_numeric[period] = {
        'model': model,
        'r2': r2,
        'mae': mae
    }
    
    print(f"\nMODELO {period.upper()} - Predição Close {period} futuro:")
    print(f"  R²: {r2:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  Intercepto: {model.intercept_:.4f}")
    print(f"  Coeficientes:")
    for i, feature in enumerate(X_numeric.columns):
        print(f"    {feature}: {model.coef_[i]:.4f}")

print(f"\n✓ 4 modelos sem dummy treinados com sucesso!")


MODELOS SEM VARIÁVEIS DUMMY

MODELO 3D - Predição Close 3d futuro:
  R²: 0.9905
  MAE: 0.3742
  Intercepto: 0.1113
  Coeficientes:
    Low: 0.7414
    High: 0.7348
    Open: -0.4835

MODELO 7D - Predição Close 7d futuro:
  R²: 0.9798
  MAE: 0.5687
  Intercepto: 0.2572
  Coeficientes:
    Low: 0.6510
    High: 0.6597
    Open: -0.3277

MODELO 15D - Predição Close 15d futuro:
  R²: 0.9637
  MAE: 0.7477
  Intercepto: 0.5417
  Coeficientes:
    Low: 0.6909
    High: 0.5792
    Open: -0.3023

MODELO 30D - Predição Close 30d futuro:
  R²: 0.9161
  MAE: 1.1824
  Intercepto: 1.0707
  Coeficientes:
    Low: 0.6751
    High: 0.4190
    Open: -0.1552

✓ 4 modelos sem dummy treinados com sucesso!


### Modelos com variáveis dummy (Low, High, Open + period_1, period_2)

In [ ]:
# Definir variáveis independentes (com dummy)
X_with_dummy = df_final[['Low', 'High', 'Open'] + dummy_columns]

print("\nMODELOS COM VARIÁVEIS DUMMY")
print("=" * 50)

# Dicionário para armazenar modelos treinados
models_with_dummy = {}

# Treinar modelo para cada janela temporal
for period, y in targets.items():
    # Divisão treino-teste (mesma partição)
    X_train, X_test, y_train, y_test = train_test_split(
        X_with_dummy, y, test_size=0.2, random_state=42)
    
    # Treinar modelo
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Fazer predições
    y_pred = model.predict(X_test)
    
    # Calcular métricas
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    # Armazenar modelo
    models_with_dummy[period] = {
        'model': model,
        'r2': r2,
        'mae': mae
    }
    
    print(f"\nMODELO {period.upper()} - Predição Close {period} futuro:")
    print(f"  R²: {r2:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  Intercepto: {model.intercept_:.4f}")
    print(f"  Coeficientes:")
    for i, feature in enumerate(X_with_dummy.columns):
        print(f"    {feature}: {model.coef_[i]:.4f}")



MODELOS COM VARIÁVEIS DUMMY

MODELO 3D - Predição Close 3d futuro:
  R²: 0.9905
  MAE: 0.3740
  Intercepto: 0.1720
  Coeficientes:
    Low: 0.7199
    High: 0.7503
    Open: -0.4871
    period_1: 0.0822
    period_2: 0.1092

MODELO 7D - Predição Close 7d futuro:
  R²: 0.9798
  MAE: 0.5711
  Intercepto: 0.3499
  Coeficientes:
    Low: 0.6019
    High: 0.6958
    Open: -0.3296
    period_1: 0.0821
    period_2: 0.1721

MODELO 15D - Predição Close 15d futuro:
  R²: 0.9636
  MAE: 0.7502
  Intercepto: 0.8348
  Coeficientes:
    Low: 0.5598
    High: 0.6747
    Open: -0.3134
    period_1: 0.3243
    period_2: 0.5362

MODELO 30D - Predição Close 30d futuro:
  R²: 0.9170
  MAE: 1.1909
  Intercepto: 1.5875
  Coeficientes:
    Low: 0.5090
    High: 0.5375
    Open: -0.1888
    period_1: 0.7453
    period_2: 0.9247

✓ 4 modelos com dummy treinados com sucesso!

RESUMO: 8 MODELOS TREINADOS
✓ 4 modelos sem variáveis dummy (Low, High, Open)
✓ 4 modelos com variáveis dummy (Low, High, Open + period_